In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import torch
from torch import nn
from tqdm import tqdm
from linformer import Linformer
from vit_pytorch.efficient import ViT
import torch.nn.functional as F
from torch.optim.lr_scheduler import StepLR
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

In [2]:
import os
os.environ['TORCH_USE_CUDA_DSA']="1"
os.environ['CUDA_LAUNCH_BLOCKING']="1"

Load the data

In [3]:
connectome_data = pd.read_csv("./dataset/widsdatathon2025/TRAIN_CORRECTED/TRAIN_FUNCTIONAL_CONNECTOME_MATRICES_new_36P_Pearson.csv")

qu_df = pd.read_excel("dataset/widsdatathon2025/TRAIN_CORRECTED/TRAIN_QUANTITATIVE_METADATA_new.xlsx")
# qu_df = qu_df.drop(index=qu_df.query('MRI_Track_Age_at_Scan >= 10 & MRI_Track_Age_at_Scan <= 10').index)
# qu_df = qu_df.drop(index=qu_df[qu_df["MRI_Track_Age_at_Scan"].isna() == True].index)

# connectome_data = connectome_data.drop(index=connectome_data[connectome_data["participant_id"].isin(qu_df["participant_id"].values) == False].index)


In [4]:
train_labels = pd.read_excel("./dataset/widsdatathon2025/TRAIN_CORRECTED/TRAINING_SOLUTIONS.xlsx")

# train_labels = train_labels.drop(index=connectome_data[connectome_data["participant_id"].isin(qu_df["participant_id"].values) == False].index)


In [5]:
# split the data into 4 splits
train_labels_adhd_male = train_labels[(train_labels["ADHD_Outcome"] == 1) & (train_labels["Sex_F"] == 0)]
train_labels_adhd_female = train_labels[(train_labels["ADHD_Outcome"] == 1) & (train_labels["Sex_F"] == 1)]
train_labels_no_adhd_male = train_labels[(train_labels["ADHD_Outcome"] == 0) & (train_labels["Sex_F"] == 0)]
train_labels_no_adhd_female = train_labels[(train_labels["ADHD_Outcome"] == 0) & (train_labels["Sex_F"] == 1)]

In [6]:
connectome_data_adhd_male = connectome_data[connectome_data['participant_id'].isin(train_labels_adhd_male['participant_id'])]
connectome_data_adhd_female = connectome_data[connectome_data['participant_id'].isin(train_labels_adhd_female['participant_id'])]
connectome_data_no_adhd_male = connectome_data[connectome_data['participant_id'].isin(train_labels_no_adhd_male['participant_id'])]
connectome_data_no_adhd_female = connectome_data[connectome_data['participant_id'].isin(train_labels_no_adhd_female['participant_id'])]

In [7]:
expected_num_regions = 200

# Build Adjacency matrices for 4 classes - ADHD (male), ADHD (female), No ADHD (male), No ADHD (female)

In [8]:
def construct_adjacency_matrix(connectome_data):
    connectivity_vectors = connectome_data.iloc[:, 1:].values
    num_patients = connectome_data.shape[0]

    upper_tri_indices = np.triu_indices(expected_num_regions, k=1)

    matrix_full = np.zeros((num_patients, expected_num_regions, expected_num_regions))

    for i in range(num_patients):
        matrix = np.zeros((expected_num_regions, expected_num_regions))
        matrix[upper_tri_indices] = connectivity_vectors[i]
        matrix += matrix.T
        np.fill_diagonal(matrix, 0)
        matrix_full[i] = matrix

    return matrix_full

In [9]:
print(connectome_data_adhd_male.shape)
print(connectome_data_adhd_male.iloc[0,0])
type(connectome_data_adhd_male)
cdam_t = connectome_data_adhd_male.drop(columns=["participant_id"]).transpose()

(392, 19901)
70z8Q2xdTXM3


In [10]:
def make_ctme_image(row:pd.Series, imageDir = "CTME_images/"):
  data = np.array([row[1:].to_numpy(dtype=np.float32)])
  data = (data+1.0)/2.0
  data = data*(data>=0.05).astype(np.float32)

  data = (data - data.min())/(data.max() - data.min())

  data = np.append(data, np.zeros(shape=(1,100), dtype=np.float32))

  data = data.reshape((200,100))

  return data


In [11]:
matrix_adhd_male = np.array([make_ctme_image(row) for index, row in connectome_data_adhd_male.iterrows()])

In [12]:
matrix_adhd_female = np.array([make_ctme_image(row) for index, row in connectome_data_adhd_female.iterrows()])

In [13]:
matrix_no_adhd_female = np.array([make_ctme_image(row) for index, row in connectome_data_no_adhd_female.iterrows()])

In [14]:
matrix_no_adhd_male = np.array([make_ctme_image(row) for index, row in connectome_data_no_adhd_male.iterrows()])

In [15]:
print(len(matrix_no_adhd_female)+len(matrix_adhd_female))
print(len(matrix_no_adhd_male)+len(matrix_adhd_male))

274
538


In [16]:
print(matrix_adhd_male.shape)
print(matrix_adhd_female.shape)
print(matrix_no_adhd_male.shape)
print(matrix_no_adhd_female.shape)

(392, 200, 100)
(164, 200, 100)
(146, 200, 100)
(110, 200, 100)


In [17]:
# combined_matrix_dataset = list(matrix_adhd_female) + list(matrix_adhd_male) + list(matrix_no_adhd_male) + list(matrix_no_adhd_female)

combined_matrix_dataset = list(matrix_adhd_female) + list(matrix_no_adhd_female) + list(matrix_adhd_male) + list(matrix_no_adhd_male)

combined_matrix_dataset = np.array(combined_matrix_dataset)

In [18]:
labels = np.array([1] * (164+110) + [0] * (392+146)).astype(np.float32)
# labels = np.array(labels).astype(np.float32)


Dataset Class

In [19]:
class ConnectomeMapDataset(Dataset):
    def __init__(self, files, labels):
        self.files = files
        self.labels = labels

    def __len__(self):
        self.filelength = len(self.files)
        return self.filelength

    def __getitem__(self, idx):
        file = self.files[idx]
        file = file.reshape((1, file.shape[0], file.shape[1]))
        label = self.labels[idx]

        return file, label


In [20]:
train_list, test_list, train_labels, test_labels = train_test_split(combined_matrix_dataset, labels, test_size=0.2, random_state=42, shuffle=True)


In [21]:
train_data = ConnectomeMapDataset(train_list, train_labels)
test_data = ConnectomeMapDataset(test_list, test_labels)

In [22]:
# Training settings
batch_size = 64
epochs = 100
lr = 3e-5
gamma = 0.7
seed = 42

In [23]:
train_loader = DataLoader(dataset = train_data, batch_size=batch_size, shuffle=True )
test_loader = DataLoader(dataset = test_data, batch_size=batch_size, shuffle=True)

In [24]:
print(len(train_data), len(train_loader))


649 11


In [25]:
print(len(test_data), len(test_loader))

163 3


# Model

In [26]:
import matplotlib.pyplot as plt

def makeAccGraph(stats):
  fig, ax = plt.subplots(1, 2, figsize=(8,4))
  ax[0].set_title("Accuracy")
  ax[0].plot(stats["Accuracy"], label="Training")
  ax[0].plot(stats["Val Accuracy"], label="Validation")
  ax[0].legend()
  ax[0].set_xlabel("Epoch")
  ax[0].set_ylabel("Accuracy")

  ax[1].legend()
  ax[1].plot(stats["Loss"], label="Training")
  ax[1].plot(stats["Val Loss"], label="Validation")
  ax[1].set_xlabel("Epoch")
  ax[1].set_title("Loss")
  ax[1].set_ylabel("Loss")
  plt.legend()
  plt.savefig("training-graph.png")
  plt.close()


In [27]:
device = 'cuda'

In [28]:
efficient_transformer = Linformer(
    dim=128,
    seq_len=400+1,  # 7x7 patches + 1 cls-token
    depth=12,
    heads=8,
    k=64
)

In [29]:
model = ViT(
    dim=128,
    image_size=200,
    patch_size=10,
    num_classes=1,
    transformer=efficient_transformer,
    channels=1,
).to(device)

In [30]:
# loss function
criterion = nn.BCELoss()
# optimizer
optimizer = optim.Adam(model.parameters(), lr=lr)
# scheduler
scheduler = StepLR(optimizer, step_size=1, gamma=gamma)

In [31]:
epoch_acc_l = []
epoch_loss_l = []
epoch_val_acc_l = []
epoch_val_loss_l = []



for epoch in range(epochs):
    epoch_loss = 0
    epoch_accuracy = 0

    for data, label in tqdm(train_loader):
        data = data.to(device)
        label = label.to(device)
        # label = label.unsqueeze(dim=1)


        output = model(data)
        output = output.squeeze()
        # print(output.shape)
        # print(output.dtype)

        loss = criterion(output, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        output = (output>0.5).type(torch.float32)

        # acc = (output.argmax(dim=1) == label).float().mean()
        acc = (output == label).float().mean()
        epoch_accuracy += acc / len(train_loader)
        epoch_loss += loss / len(train_loader)

    with torch.no_grad():
        epoch_val_accuracy = 0
        epoch_val_loss = 0
        for data, label in test_loader:
            data = data.to(device)
            label = label.to(device)
            # label = label.unsqueeze(dim=1)

            val_output = model(data)
            val_output = val_output.squeeze()

            # print(val_output)


            val_loss = criterion(val_output, label)
            val_output = (val_output > 0.5).type(torch.float32)

            acc = (val_output == label).float().mean()
            # acc = (val_output.argmax(dim=1) == label).float().mean()
            epoch_val_accuracy += acc / len(test_loader)
            epoch_val_loss += val_loss / len(test_loader)

    epoch_acc_l += [float(epoch_accuracy)]
    epoch_loss_l += [float(epoch_loss)]
    epoch_val_loss_l += [float(epoch_val_loss)]
    epoch_val_acc_l += [float(epoch_val_accuracy)]

    training_stats = pd.DataFrame({"Accuracy":epoch_acc_l, "Loss":epoch_loss_l, "Val Accuracy":epoch_val_acc_l, "Val Loss":epoch_val_loss_l})

    makeAccGraph(training_stats)
    print(
        f"Epoch : {epoch+1} - loss : {epoch_loss:.4f} - acc: {epoch_accuracy:.4f} - val_loss : {epoch_val_loss:.4f} - val_acc: {epoch_val_accuracy:.4f}\n"
    )


100%|██████████| 11/11 [00:01<00:00,  9.88it/s]
/tmp/ipykernel_195826/2383425895.py:12: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax[1].legend()


Epoch : 1 - loss : 0.7062 - acc: 0.5732 - val_loss : 0.6472 - val_acc: 0.6731



100%|██████████| 11/11 [00:00<00:00, 13.27it/s]


Epoch : 2 - loss : 0.6497 - acc: 0.6499 - val_loss : 0.6380 - val_acc: 0.6817



100%|██████████| 11/11 [00:00<00:00, 12.86it/s]


...

Epoch : 98 - loss : 0.0018 - acc: 1.0000 - val_loss : 1.6583 - val_acc: 0.6765



100%|██████████| 11/11 [00:00<00:00, 12.95it/s]


Epoch : 99 - loss : 0.0018 - acc: 1.0000 - val_loss : 1.5635 - val_acc: 0.6851



100%|██████████| 11/11 [00:00<00:00, 12.93it/s]


Epoch : 100 - loss : 0.0018 - acc: 1.0000 - val_loss : 1.6819 - val_acc: 0.6713



# Predictions

In [32]:
submission_test_connectome_og = pd.read_csv("dataset/widsdatathon2025/TEST/TEST_FUNCTIONAL_CONNECTOME_MATRICES.csv")
# submission_test = pd.read_excel("dataset/widsdatathon2025/TEST/TEST_FUNCTIONAL_CONNECTOME_MATRICES.csv")

In [33]:
print(submission_test_connectome_og.keys())

Index(['participant_id', '0throw_1thcolumn', '0throw_2thcolumn',
       '0throw_3thcolumn', '0throw_4thcolumn', '0throw_5thcolumn',
       '0throw_6thcolumn', '0throw_7thcolumn', '0throw_8thcolumn',
       '0throw_9thcolumn',
       ...
       '195throw_196thcolumn', '195throw_197thcolumn', '195throw_198thcolumn',
       '195throw_199thcolumn', '196throw_197thcolumn', '196throw_198thcolumn',
       '196throw_199thcolumn', '197throw_198thcolumn', '197throw_199thcolumn',
       '198throw_199thcolumn'],
      dtype='object', length=19901)


In [34]:
submission_test_connectome = np.array([make_ctme_image(row) for index, row in submission_test_connectome_og.iterrows()])

In [35]:
print(submission_test_connectome.shape)

(304, 200, 100)


In [36]:
sub_test_data = ConnectomeMapDataset(submission_test_connectome, [0]*len(submission_test_connectome))
sub_test_loader = DataLoader(dataset = sub_test_data, batch_size=batch_size, shuffle=False)


In [46]:
test_outputs = []

model.eval()
with torch.no_grad():

    for data, label in tqdm(sub_test_loader):
        data = data.to(device)

        val_output = model(data)
        val_output = val_output.squeeze()

        val_output = (val_output > 0.5).type(torch.float32)

        test_outputs += [val_output.detach().cpu().numpy()]


100%|██████████| 5/5 [00:00<00:00, 31.96it/s]


In [ ]:
better_outputs = []

for arr in test_outputs:
  for i in arr:
    better_outputs += [i]


In [48]:
print(test_outputs[0].shape)

(64,)


In [ ]:
print(len(better_outputs))
outputs = pd.DataFrame({"participant_id":submission_test_connectome_og["participant_id"], "Sex_F":better_outputs})

# outputs.to_excel("OUTPUTS.xlsx", index=False)

304
